# Laboratorio # 5

Vamos a leer el dataset

In [21]:
import pandas as pd


train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")


In [23]:
train.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [24]:
test.head()

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan


Vamos a empezar a limpiar el dataset. Primero vamos a convertir todo el texto a minúscula.

In [25]:
import re

def convert_to_lower(word):
    return word.lower()

def remove_special_characters(word):
    return word.replace("#", "").replace("@", "").replace("'", "")

def remove_url(word):
    text = re.sub(r'http\S+|www\S+|https\S+', '', word)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def remove_emojis(word):
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # emoticonos 🙂
        "\U0001F300-\U0001F5FF"  # símbolos y pictogramas 🗻
        "\U0001F680-\U0001F6FF"  # transporte y mapas 🚗
        "\U0001F1E0-\U0001F1FF"  # banderas 🇬🇹
        "\U00002700-\U000027BF"  # símbolos varios ➡
        "\U0001F900-\U0001F9FF"  # suplementarios 🤖
        "\U00002600-\U000026FF"  # misceláneos ☀
        "\U00002B00-\U00002BFF"  # flechas ⬆
        "\U0001FA70-\U0001FAFF"  # emojis recientes 🪐
        "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.sub(r'', word).strip()

train['text'] = train['text'].astype(str).apply(convert_to_lower)
train['text'] = train['text'].astype(str).apply(remove_special_characters)
train['text'] = train['text'].astype(str).apply(remove_url)
train['text'] = train['text'].astype(str).apply(remove_emojis)


train.head()

,id,keyword,location,text,target
0,1,NaN,NaN,our deeds are the reason of this earthquake ma...,1
1,4,NaN,NaN,forest fire near la ronge sask. canada,1
2,5,NaN,NaN,all residents asked to shelter in place are be...,1
3,6,NaN,NaN,"13,000 people receive wildfires evacuation ord...",1
4,7,NaN,NaN,just got sent this photo from ruby alaska as s...,1


## Limpieza
### Quitar los signos de puntuacion


In [26]:
import re

train["text"] = train["text"].str.replace(r'[^\w\s]', '', regex=True)

test["text"] = test["text"].str.replace(r'[^\w\s]', '', regex=True)


### Quitar los artículos, preposiciones y conjunciones


In [27]:
import pandas as pd
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    tokens = text.split()  # separar en palabras
    tokens = [word for word in tokens if word.lower() not in stop_words]
    return " ".join(tokens)

# Aplicar a las columnas
train["text"] = train["text"].apply(remove_stopwords)
test["text"] = test["text"].apply(remove_stopwords)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ppguz\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


### Quitar numeros

Primero vamos a ver cuales eliminar, listando por los que mas aparecen


In [29]:
import re
import pandas as pd

def extract_numbers(text):
    return re.findall(r'\d+', str(text))
train_nums = train.copy()
train_nums["numbers"] = train_nums["text"].apply(extract_numbers)

test_nums = test.copy()
test_nums["numbers"] = test_nums["text"].apply(extract_numbers)

all_nums = pd.concat([train_nums[["numbers"]], test_nums[["numbers"]]], ignore_index=True)

all_nums = all_nums[all_nums["numbers"].map(len) > 0]

import numpy as np
all_numbers_flat = np.concatenate(all_nums["numbers"].values)

summary = pd.Series(all_numbers_flat).value_counts().reset_index()
summary.columns = ["number", "count"]
summary


,number,count
0,2,641
1,3,504
2,4,468
3,1,462
4,5,461
...,...,...
838,1233,1
839,801,1
840,317,1
841,4552,1


Sabiendo esto vamos a conservar los numeros 1945, 911, 2008, 2014, 1980, 2013, 2016, 2011  ya que son fechas importantes, ademas de que son numeros de telefono que pueden tener relacion con el analisis de sentimientos


In [30]:
import re

allowed_numbers = {"1945", "911", "2008", "2014", "1980", "2013", "2016", "2011"}

def remove_unwanted_numbers(text):
    return re.sub(r'\b(?!' + '|'.join(allowed_numbers) + r')\d+\b', '', str(text))

train["text"] = train["text"].apply(remove_unwanted_numbers)
test["text"] = test["text"].apply(remove_unwanted_numbers)


test.head()



,id,keyword,location,text
0,0,NaN,NaN,happened terrible car crash
1,2,NaN,NaN,Heard earthquake different cities stay safe ev...
2,3,NaN,NaN,forest fire spot pond geese fleeing across str...
3,9,NaN,NaN,Apocalypse lighting Spokane wildfires
4,11,NaN,NaN,Typhoon Soudelor kills China Taiwan


In [34]:
train.head()

,id,keyword,location,text,target
0,1,NaN,NaN,deeds reason earthquake may allah forgive us,1
1,4,NaN,NaN,forest fire near la ronge sask canada,1
2,5,NaN,NaN,residents asked shelter place notified officer...,1
3,6,NaN,NaN,people receive wildfires evacuation orders ca...,1
4,7,NaN,NaN,got sent photo ruby alaska smoke wildfires pou...,1


Ahora vamos a contar las palabras más frecuentes en cada categoría. Primero vamos a empezar con los tweets que si hablan de desastres naturales

In [46]:
from collections import Counter

natural_disaters = train[train['target'] == 1]

def count_words(dataset, column):
    all_words = ''
    for i in dataset[column]:
        all_words = all_words + i + ' '

    all_words = all_words.split()

    frequency = Counter(all_words)

    print("{:<10} {:<10}".format("Palabra", "Frecuencia"))
    print("-" * 20)
    for palabra, freq in frequency.most_common():
        print("{:<10} {:<10}".format(palabra, freq))


count_words(natural_disaters, 'text')

Palabra    Frecuencia
--------------------
fire       178       
news       136       
via        121       
disaster   117       
california 111       
suicide    110       
police     107       
amp        106       
people     105       
killed     93        
like       92        
hiroshima  86        
storm      85        
crash      84        
fires      84        
us         81        
families   81        
train      79        
emergency  76        
buildings  75        
bomb       74        
two        71        
mh370      71        
nuclear    70        
attack     69        
video      69        
wildfire   69        
get        66        
accident   66        
bombing    66        
one        65        
northern   64        
burning    64        
dead       63        
pm         62        
legionnaires 62        
bomber     60        
homes      58        
car        57        
still      57        
war        57        
im         56        
new        56        
atomic   

Ahora vamos con los tweets que no son de desastres naturales

In [45]:
no_natural_disasters = train[train['target'] == 0]
count_words(no_natural_disasters, 'text')

Palabra    Frecuencia
--------------------
like       253       
im         243       
amp        192       
new        168       
get        163       
dont       141       
one        127       
body       112       
via        99        
would      97        
video      96        
people     91        
love       89        
know       85        
back       84        
time       83        
us         83        
got        83        
see        82        
cant       81        
emergency  81        
full       81        
day        78        
youtube    76        
going      75        
still      72        
fire       72        
go         67        
want       67        
good       67        
think      66        
man        62        
world      62        
lol        61        
rt         60        
life       60        
u          59        
youre      58        
first      58        
news       57        
last       56        
burning    56        
really     55        
way        